In [ ]:
from common import *

## 4. Učitavanje i početno formatiranje podataka

Pošto smo već spomenuli da će svaka transformacija skupa podataka biti čuvana u zasebnom backup dataframe-u, valjalo bi kreirati funkciju koja će za prosleđenu putanju do csv fajla učitati podatke. 

U sklopu ovog dela, pored učitavanja podataka, izvršićemo i neophodnu pripremu podataka za sve buduće analize. Ova priprema podrazumeva da ćemo obezbediti da se datum tretira kao objekat tipa DateTime a ne kao string, takođe izdvojićemo svaki deo datuma (mesec, dan, godinu kao i dan u godini) u zasebnu kolonu. Kako bismo podatke u daljoj analizi posmatrali kao vremensku seriju, moramo obezbediti da sve vrste budu sortirane po lokaciji i datumu.

In [ ]:
data = loadData('InputData/weatherAUS.csv')

old_df = data.copy()
data = data.copy()
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['DayOfYear'] = data['Date'].dt.dayofyear
write_log(data, "Datum tranformisan u 4 nove promenljive: godina, mesec, dan i dan u godini.", "weatherAUSAfter4.csv")

## 5. Detekcija strukturalnih grešaka

Pre bilo kakve eksplorativne analize, potrebno je da uklonimo strukturalne greške, odnosno greške nastale pogrešnim unosom podataka.

Detekciju ovakvih grešaka možemo podeliti na sledeće dve celine:

### 5.1 Greške unosa kategorijskih promenljivih

Primetimo da u našem datasetu postoji određen broj kategorijskih promenljivih sa jasno definisanim mogućim vrednostima, u pitanju su:
1. WindGustDir, WindDir9am, WindDir3pm - smerovi vetra sa 16 mogućih vrednosti (N, NNE, NE, ENE, E, ESE, SE, SSE, S, SSW, SW, WSW, W, WNW, NW, NNW)
2. Cloud9am, Cloud3pm - Pokrivenost površine neba oblacima, ova vrednost se izražava u oktama (osminama neba prekrivenog oblacima), intuitivno je jasno da mogući raspon ovih vrednosti predstavljaju celi brojevi u intervalu od 0 do 8, specijalno vrednost 9 označava da merenje nije bilo moguće zbog magle.

Možemo izvršiti analizu da li je došlo do strukturalne anomalije, odnosno da li je pri unosu ovih vrednosti došlo do greške u kucanju, u zavisnosti od tipova ove greske (da li je moguće prepoznati koja vrednost je trebala biti uneta) zaključićemo kao dalje da tretiramo ove anomalije.

In [ ]:
valid_wind_dirs = {
    "N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
    "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"
}

valid_cloud_values = set(range(0, 10))

wind_columns = ["WindGustDir", "WindDir9am", "WindDir3pm"]
cloud_columns = ["Cloud9am", "Cloud3pm"]

print("=== Detekcija anomalija u kategorijskim kolonama ===\n")

for col in wind_columns:
    invalid = data[~data[col].isin(valid_wind_dirs)][col]
    if len(invalid) > 0:
        print(f"Kolona: {col}")
        print(f"Broj anomalnih vrednosti: {len(invalid)}")
        print(f"Jedinstvene anomalne vrednosti: {invalid.unique()}\n")
    else:
        print(f"Kolona: {col} - nema anomalija.\n")

for col in cloud_columns:
    invalid = data[~data[col].isin(valid_cloud_values)][col]
    if len(invalid) > 0:
        print(f"Kolona: {col}")
        print(f"Broj anomalnih vrednosti: {len(invalid)}")
        print(f"Jedinstvene anomalne vrednosti: {invalid.unique()}\n")
    else:
        print(f"Kolona: {col} - nema anomalija.\n")

write_log(data, "Detekcija kategorijskih anomalija", "weatherAusAfter5_1.csv")



Primećujemo da, ako izuzmemo nedostajuće vrednosti, ne postoje greške u unosu kategorijskih promenljivih.

### 5.2 Fizičke greške

Potrebno je utvrditi da li u dataset-u postoje podaci koji krše zakone fizike i koji su očigledne anomalije. Ako ovakvi podaci postoje potrebno ih je eliminisati u samom početku, kako bi dalja analiza i treniranje modela bili jednostavniji i precizniji.

Fizička pravila koja proveravamo su sledeća:

1. Da li je minimalna dnevna temperatura manja od maksimalne dnevne temperature
2. Da li izmerene temperature u 9 i 15 časova pripadaju intervalu (`MinTemp`, `MaxTemp`)
3. Da li su `RainToday` i `Rainfall` u koliziji (da li jedno naglašava da je bilo kiše, a drugo ne)
4. Da li postoje dva uzastopna datuma takva da je RainTommorow(t) != RainToday(t+1)
5. Da li postoje duplirani podaci o merenju neke stanice za određeni datum
6. Da li je neka od izmerenih vrednosti veća od rekordnih vrednosti za Australiju (kako bismo izbegli brisanje noviteta sve velicine ćemo blago uvećati od rekordnih vrednosti)
7. Da li je brzina najjačeg vetra manja od brzine vetra u 9 i 15 časova
8. U meteorologiji se definiše "tačka rose" - temperatura do koje se vazduh mora ohladiti da bi postao zasićen vodom, po fizičkom zakonu ova vrednost mora biti manja od trenutne temperature, ako ovaj uslov nije ispunjen, podatke ćemo tretirati kao anomalije.
9. Da li postoji fizički nemoguća promena pritiska vazduha, veća od 2 hPa po satu
10. U ovoj fazi detekovaćemo i nedostajuće vrste u vremenskoj seriji, koje mogu ugroziti rad modela

<b>Napomena: </b> Kao referencu za "hardcode" ograničenja koristimo zvanične podatke sa sajta https://www.bom.gov.au/climate/extreme/records.shtml

U nastavku ćemo izvršiti detekciju gore navedenih anomalija. Sve anomalije koje pronađemo ćemo obrisati i postaviti vrednost NA kako bi korigovane vrednosti bile upisane u procesu popunjavanja nedostajućih vrednosti. Takođe, sve registrovane fizičke anomalije ćemo upisati u zaseban csv fajl radi dalje kontrole.

In [ ]:
OGRANICENJA_VREDNOSTI: Dict[str, Tuple[float, float]] = {
    "MinTemp":       (-15.0, 45.0),
    "MaxTemp":       (-10.0, 55.0),
    "Temp9am":       (-15.0, 50.0),
    "Temp3pm":       (-12.0, 55.0),
    "Rainfall":      (0.0, 600.0),
    "Evaporation":   (0.0, 30.0),
    "WindGustSpeed": (0.0, 200.0),
    "WindSpeed9am":  (0.0, 150.0),
    "WindSpeed3pm":  (0.0, 150.0),
    "Humidity9am":   (0.0, 100.0),
    "Humidity3pm":   (0.0, 100.0),
    "Pressure9am":   (950.0, 1060.0),
    "Pressure3pm":   (950.0, 1060.0),
    "Cloud9am":      (0.0, 9.0),
    "Cloud3pm":      (0.0, 9.0),
}

def dew_point(temp_c: pd.Series, rh_pct: pd.Series) -> pd.Series:
    a, b = 17.625, 243.04
    rh = rh_pct.clip(lower=1.0, upper=100.0) / 100.0
    alpha = np.log(rh) + (a * temp_c) / (b + temp_c)
    return (b * alpha) / (a - alpha)

errors: Dict[int, List[str]] = {}


# R1
mask = data["MinTemp"] > data["MaxTemp"]
data.loc[mask, ["MinTemp", "MaxTemp"]] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("MinTemp > MaxTemp")

# R2
tol = 0.6
for t in ("Temp9am", "Temp3pm"):
    mask = (data[t] < data["MinTemp"] - tol) | \
            (data[t] > data["MaxTemp"] + tol)
    data.loc[mask, t] = np.nan
    for idx in data.index[mask]:
        errors.setdefault(idx, []).append(f"{t} van opsega MinTemp-MaxTemp")

# R3 - usklađivanje RainToday sa Rainfall
expected = np.where(data["Rainfall"] >= 1.0, "Yes", "No")
mask = data["RainToday"] != expected
data.loc[mask, "RainToday"] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("RainToday inconsistent sa Rainfall")

# R4
nxt_today = data.groupby("Location")["RainToday"].shift(-1)
nxt_date = data.groupby("Location")["Date"].shift(-1)
contiguous = (nxt_date - data["Date"]).dt.days.eq(1)
mask = contiguous & (data["RainTomorrow"] != nxt_today)
data.loc[mask, "RainTomorrow"] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("RainTomorrow inconsistent sa RainToday sledeceg dana")

# R5 - duplikati
mask = data.duplicated(subset=["Location", "Date"], keep="first")
data.loc[mask, :] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("Dupli zapis")

# R6 - opsezi
for v, (low, high) in OGRANICENJA_VREDNOSTI.items():
    mask = (data[v] < low) | (data[v] > high)
    data.loc[mask, v] = np.nan
    for idx in data.index[mask]:
        errors.setdefault(idx, []).append(f"{v} van opsega [{low}, {high}]")

# R7
mx = data[["WindSpeed9am", "WindSpeed3pm"]].max(axis=1)
mask = (data["WindGustSpeed"] < mx).fillna(False)
data.loc[mask, "WindGustSpeed"] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("WindGustSpeed < max(WindSpeed9am, WindSpeed3pm)")

# R8
for t, h in (("Temp9am", "Humidity9am"), ("Temp3pm", "Humidity3pm")):
    td = dew_point(data[t], data[h])
    mask = (td > data[t]).fillna(False)
    data.loc[mask, t] = np.nan
    for idx in data.index[mask]:
        errors.setdefault(idx, []).append(f"Dew point > {t}")

# R9
mask = ((data["Pressure3pm"] - data["Pressure9am"]).abs() > 12.0).fillna(False)
data.loc[mask, ["Pressure9am", "Pressure3pm"]] = np.nan
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("Nedozvoljeni skok pritiska")

# R10
gap = data.groupby("Location")["Date"].diff().dt.days
mask = gap > 1
for idx in data.index[mask]:
    errors.setdefault(idx, []).append("Nedostaje dan u seriji")

# Export loga
error_df = pd.DataFrame([
    {"Index": idx, "Errors": "; ".join(msgs)}
    for idx, msgs in errors.items()
])
write_log(data,"Uklonjene fizicke anomalije", "weatherAusAfter5_2.csv")